In [3]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Paths — adjust based on actual Kaggle input folder names ──────────────
FACES140K_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'
WEIGHTS_GAN    = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-weights/deepfake_classifier.pth'
WEIGHTS_GAN_FT = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-finetuned/deepfake_classifier_finetuned.pth'
WEIGHTS_CELEB  = '/kaggle/input/datasets/dhathrikarthik/deepfake-celebdf-weights/deepfake_celebdf.pth'  # adjust path if separate dataset

# ── Rebuild test dataset (same as before) ──────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.data = []
        for label_name, label in [('real', 0), ('fake', 1)]:
            folder = os.path.join(root_dir, label_name)
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.data.append((os.path.join(folder, fname), label))
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

test_dataset = FaceDataset(f'{FACES140K_DIR}/test', transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Test set size: {len(test_dataset)}")

# ── Load all three models ───────────────────────────────────────────────────
def _load(weights_path, strip_prefix=False):
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    state = torch.load(weights_path, map_location=DEVICE)
    if strip_prefix:
        state = {k.replace('model.', '', 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model

gan_model    = _load(WEIGHTS_GAN,    strip_prefix=True)
gan_model_ft = _load(WEIGHTS_GAN_FT, strip_prefix=False)  # saved fresh via torch.save(model.state_dict(), ...), no prefix
celeb_model  = _load(WEIGHTS_CELEB,  strip_prefix=False)
print("✅ All three models loaded")

# ── Evaluate ensemble (max-fusion) on the 140k test set ────────────────────
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        probs_gan    = torch.softmax(gan_model(images),    dim=1)
        probs_gan_ft = torch.softmax(gan_model_ft(images), dim=1)
        probs_celeb  = torch.softmax(celeb_model(images),  dim=1)

        fake_scores = torch.stack([
            probs_gan[:, 1], probs_gan_ft[:, 1], probs_celeb[:, 1]
        ], dim=1)
        fake_prob_max = fake_scores.max(dim=1).values

        preds = (fake_prob_max >= 0.5).long()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Ensemble (max-fusion, 3 models) accuracy on original 140k test set: {correct/total*100:.2f}%")

Using device: cuda
Test set size: 20000
✅ All three models loaded
Ensemble (max-fusion, 3 models) accuracy on original 140k test set: 85.34%


In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Paths — adjust based on actual Kaggle input folder names ──────────────
FACES140K_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'
WEIGHTS_GAN    = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-weights/deepfake_classifier.pth'
WEIGHTS_GAN_FT = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-finetuned/deepfake_classifier_finetuned.pth'
WEIGHTS_CELEB  = '/kaggle/input/datasets/dhathrikarthik/deepfake-celebdf-weights/deepfake_celebdf.pth'  # adjust path if separate dataset

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.data = []
        for label_name, label in [('real', 0), ('fake', 1)]:
            folder = os.path.join(root_dir, label_name)
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.data.append((os.path.join(folder, fname), label))
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

test_dataset = FaceDataset(f'{FACES140K_DIR}/test', transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Test set size: {len(test_dataset)}")

def _load(weights_path, strip_prefix=False):
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    state = torch.load(weights_path, map_location=DEVICE)
    if strip_prefix:
        state = {k.replace('model.', '', 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model

gan_model    = _load(WEIGHTS_GAN,    strip_prefix=True)
gan_model_ft = _load(WEIGHTS_GAN_FT, strip_prefix=False)
celeb_model  = _load(WEIGHTS_CELEB,  strip_prefix=False)
print("✅ All three models loaded")

# ── Track confusion matrix components ───────────────────────────────────────
tp, tn, fp, fn = 0, 0, 0, 0  # fp = real called fake, fn = fake called real

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        probs_gan    = torch.softmax(gan_model(images),    dim=1)
        probs_gan_ft = torch.softmax(gan_model_ft(images), dim=1)
        probs_celeb  = torch.softmax(celeb_model(images),  dim=1)

        fake_scores = torch.stack([
            probs_gan[:, 1], probs_gan_ft[:, 1], probs_celeb[:, 1]
        ], dim=1)
        fake_prob_max = fake_scores.max(dim=1).values
        preds = (fake_prob_max >= 0.5).long()

        for pred, label in zip(preds.tolist(), labels.tolist()):
            if pred == 1 and label == 1:
                tp += 1
            elif pred == 0 and label == 0:
                tn += 1
            elif pred == 1 and label == 0:
                fp += 1  # real wrongly called fake
            elif pred == 0 and label == 1:
                fn += 1  # fake wrongly called real

total = tp + tn + fp + fn
print(f"\nConfusion matrix breakdown (ensemble, max-fusion):")
print(f"  True Positives  (fake→fake, correct): {tp}")
print(f"  True Negatives  (real→real, correct): {tn}")
print(f"  False Positives (real→fake, WRONG):   {fp}")
print(f"  False Negatives (fake→real, WRONG):   {fn}")
print(f"\nAccuracy: {(tp+tn)/total*100:.2f}%")
print(f"False Positive Rate (real wrongly flagged): {fp/(fp+tn)*100:.2f}%")
print(f"False Negative Rate (fake wrongly missed):  {fn/(fn+tp)*100:.2f}%")

Using device: cuda
Test set size: 20000
✅ All three models loaded

Confusion matrix breakdown (ensemble, max-fusion):
  True Positives  (fake→fake, correct): 9988
  True Negatives  (real→real, correct): 7081
  False Positives (real→fake, WRONG):   2919
  False Negatives (fake→real, WRONG):   12

Accuracy: 85.34%
False Positive Rate (real wrongly flagged): 29.19%
False Negative Rate (fake wrongly missed):  0.12%
